In [22]:
import pandas as pd

df = pd.read_csv(r'C:\Users\karti\Desktop\IKEA-Analytics\RetailCleaned.csv', header=None, low_memory=False)
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'SourceSheet']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [23]:
df.shape

(1033036, 9)

In [24]:
df.info

<bound method DataFrame.info of          Invoice StockCode                         Description  Quantity  \
0         489434     21232      STRAWBERRY CERAMIC TRINKET BOX        24   
1         489434     21523  FANCY FONT HOME SWEET HOME DOORMAT        10   
2         489434     21871                 SAVE THE PLANET MUG        24   
3         489434     22041        RECORD FRAME 7" SINGLE SIZE         48   
4         489434     22064          PINK DOUGHNUT TRINKET POT         24   
...          ...       ...                                 ...       ...   
1033031  C581490     23144     ZINC T-LIGHT HOLDER STARS SMALL       -11   
1033032  C581499         M                              Manual        -1   
1033033  C581568     21258          VICTORIAN SEWING BOX LARGE        -5   
1033034  C581569     20979       36 PENCILS TUBE RED RETROSPOT        -5   
1033035  C581569     84978    HANGING HEART JAR T-LIGHT HOLDER        -1   

                InvoiceDate   Price  Customer_ID       

In [25]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY

rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()

snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()


In [26]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'), # method='first' if we get same freq 2 times, the 1st occurence gets earlier rank and 2nd occ gets next rank
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


def segment_customer(row): #goes through each row value of every col
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'

    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost' 


rfm['Segment'] = rfm.apply(segment_customer, axis =1) #axis=1 => each row of table is passed automatically by pandas as a parameter in segment_customer fn
rfm['IsChurned'] = rfm['Recency'] > 90

In [27]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
 
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

df['Revenue'] = df['Quantity'] * df['Price']


df_final = df.merge(
    rfm[[
        'Customer_ID', 'Recency', 'Frequency', 'Monetary',
        'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned'
    ]],
    on='Customer_ID',
    how='left'
)


df_final.to_excel(r'C:\Users\karti\Desktop\IKEA-Analytics\EcommerceDashboardData.xlsx', index=False)